In [ ]:
# 6) Identifique e trate inconsistências na base (campos nulos, valores fora do padrão). Documente o que foi encontrado e como você tratou cada caso.

import pandas as pd
from pathlib import Path
from IPython.display import display

DATA_DIR = Path('../data')

print("Iniciando Pipeline de Qualidade de Dados (ETL)...\n")

tabelas_raw = {
    'clientes':        pd.read_csv(DATA_DIR / 'clientes.csv'),
    'produtos':        pd.read_csv(DATA_DIR / 'produtos.csv'),
    'pedidos':         pd.read_csv(DATA_DIR / 'pedidos.csv'),
    'itens_pedido':    pd.read_csv(DATA_DIR / 'itens_pedido.csv'),
    'avaliacoes':      pd.read_csv(DATA_DIR / 'avaliacoes.csv'),
    'tickets_suporte': pd.read_csv(DATA_DIR / 'tickets_suporte.csv'),
}

log_auditoria = []

for nome, df in tabelas_raw.items():
    log_tabela = []

    n_dup = df.duplicated().sum()
    if n_dup > 0:
        df.drop_duplicates(inplace=True)
        log_tabela.append(f"Removidas {n_dup} linhas duplicadas.")

    if nome == 'pedidos':
        n_sem_valor = df['valor_total'].isnull().sum()
        if n_sem_valor:
            df.dropna(subset=['valor_total'], inplace=True)
            log_tabela.append(f"Removidos {n_sem_valor} pedidos sem 'valor_total'.")

        df['data_pedido'] = pd.to_datetime(df['data_pedido'], errors='coerce')
        n_data_inv = df['data_pedido'].isnull().sum()
        if n_data_inv:
            df.dropna(subset=['data_pedido'], inplace=True)
            log_tabela.append(f"Removidos {n_data_inv} pedidos com data corrompida.")

    elif nome == 'itens_pedido':
        n_desc = df['desconto_aplicado'].isnull().sum()
        if n_desc:
            df['desconto_aplicado'] = df['desconto_aplicado'].fillna(0.0)
            log_tabela.append(f"Imputado 0.0 em {n_desc} itens sem 'desconto_aplicado'.")

        n_qtd  = (df['quantidade']      < 0).sum()
        n_prec = (df['preco_praticado'] < 0).sum()
        if n_qtd or n_prec:
            df['quantidade']      = df['quantidade'].abs()
            df['preco_praticado'] = df['preco_praticado'].abs()
            log_tabela.append(f"Corrigidos via abs(): {n_qtd} quantidades e {n_prec} preços negativos.")

    elif nome == 'avaliacoes':
        n_sem_com = df['comentario'].isnull().sum()
        if n_sem_com:
            df['comentario'] = df['comentario'].fillna('Sem comentário')
            log_tabela.append(f"Preenchidos {n_sem_com} comentários nulos.")

    elif nome == 'tickets_suporte':
        n_abertos = df['data_resolucao'].isnull().sum()
        if n_abertos:
            log_tabela.append(f"Mantidos {n_abertos} nulos em 'data_resolucao' (tickets abertos válidos).")

    elif nome == 'clientes':
        if 'canal_aquisicao' in df.columns:
            canal_map_etl = {
                'orgânico':     'organico',
                'paid_search':  'trafego_pago',
                'redes_sociais':'redes_sociais',
                'indicação':    'indicacao',
                'indicacao':    'indicacao',
            }
            df['canal_aquisicao'] = df['canal_aquisicao'].str.strip().str.lower().replace(canal_map_etl)
            log_tabela.append("Canal de aquisição normalizado (snake_case).")
        else:
            log_tabela.append("Coluna 'canal_aquisicao' não encontrada para normalização.")

    for col in df.select_dtypes(include=['object', 'string']).columns:
        df[col] = df[col].str.strip()

    if 'status' in df.columns:
        df['status'] = df['status'].str.lower()
        if nome == 'tickets_suporte':
            mapa_status = {'resolvido': 'fechado', 'escalado': 'em_andamento'}
            n_leg = df['status'].isin(mapa_status.keys()).sum()
            if n_leg:
                df['status'] = df['status'].replace(mapa_status)
                log_tabela.append(f"Status padronizados: {n_leg} registros legados renomeados.")

    out_path = DATA_DIR / f'{nome}_limpo.csv'
    df.to_csv(out_path, index=False)

    log_tabela.append("Strip aplicado em colunas de texto.")
    
    log_auditoria.append({
        "Tabela Processada": nome.upper(),
        "Resumo de Alterações e Tratamentos": " | ".join(log_tabela)
    })

df_relatorio = pd.DataFrame(log_auditoria)

print("=" * 80)
print("RELATÓRIO DE AUDITORIA E TRATAMENTOS (DATA QUALITY)")
print("=" * 80)
estilo = df_relatorio.style \
    .set_properties(**{'text-align': 'left'}) \
    .set_table_styles([{'selector': 'th', 'props': [('text-align', 'left')]}])

display(estilo)

print("\n" + "=" * 80)
print("ETL finalizado. Arquivos '_limpo.csv' gerados com sucesso e prontos para uso.")
print("=" * 80)

Iniciando Pipeline de Qualidade de Dados (ETL)...

RELATÓRIO DE AUDITORIA E TRATAMENTOS (DATA QUALITY)


,Tabela Processada,Resumo de Alterações e Tratamentos
0,CLIENTES,Canal de aquisição normalizado (snake_case). | Strip aplicado em colunas de texto.
1,PRODUTOS,Strip aplicado em colunas de texto.
2,PEDIDOS,Removidos 79 pedidos sem 'valor_total'. | Strip aplicado em colunas de texto.
3,ITENS_PEDIDO,Imputado 0.0 em 301 itens sem 'desconto_aplicado'. | Strip aplicado em colunas de texto.
4,AVALIACOES,Preenchidos 1679 comentários nulos. | Strip aplicado em colunas de texto.
5,TICKETS_SUPORTE,Mantidos 1653 nulos em 'data_resolucao' (tickets abertos válidos). | Status padronizados: 3194 registros legados renomeados. | Strip aplicado em colunas de texto.



ETL finalizado. Arquivos '_limpo.csv' gerados com sucesso e prontos para uso.
